# Claude OS — Context Efficiency Curve
**Phase 0 analysis.** Plots the quality proxy signal against context utilisation for each session,
and detects the inflection point where quality begins to degrade.

Run the harness first:
```bash
cd packages/core
bun run harness --name my-session --turns 20
bun run export
```

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy.signal import savgol_filter
from scipy.stats import pearsonr

plt.rcParams.update({
    'figure.facecolor': '#0d0d0f',
    'axes.facecolor': '#1c1c1e',
    'axes.edgecolor': '#3a3a3c',
    'axes.labelcolor': '#aeaeb2',
    'xtick.color': '#636366',
    'ytick.color': '#636366',
    'text.color': '#f2f2f7',
    'grid.color': '#2c2c2e',
    'grid.linewidth': 0.5,
    'font.family': 'monospace',
    'font.size': 10,
})

GREEN = '#34c759'
AMBER = '#ff9500'
RED   = '#ff3b30'
DIM   = '#636366'

In [2]:
import json

SESSION_DIR = Path('sessions')

# ── Filters ──────────────────────────────────────────────────────────────────
# The exporter dumps every session (138+). Plotting them all yields an unreadable
# grid, so pick the sessions worth looking at. Tune these three knobs:
MIN_TURNS    = 30      # drop short sessions — too few turns to show a degradation curve
MAX_SESSIONS = 12      # cap the grid; highest-turn sessions are kept first
NAME_FILTER  = None    # substring match on session name, e.g. 'finance'; None = keep all


def load_session(csv_path: Path):
    """Load a session CSV plus the name/model from its JSON sidecar."""
    df = pd.read_csv(csv_path)
    meta = {}
    json_path = csv_path.with_suffix('.json')
    if json_path.exists():
        meta = json.loads(json_path.read_text()).get('session', {})
    return df, (meta.get('name') or csv_path.stem)


csv_files = sorted(SESSION_DIR.glob('*.csv'))
candidates = []
for f in csv_files:
    df, name = load_session(f)
    if len(df) < MIN_TURNS:
        continue
    if NAME_FILTER and NAME_FILTER.lower() not in name.lower():
        continue
    candidates.append((f.stem, name, len(df), df))

# Highest-turn sessions first, then cap to MAX_SESSIONS.
candidates.sort(key=lambda c: c[2], reverse=True)
candidates = candidates[:MAX_SESSIONS]

# Keyed by "name · slug" so duplicate session names stay distinct in the grid.
dfs = {f'{name} · {sid[:8]}': df for sid, name, _, df in candidates}

print(f'{len(csv_files)} CSVs on disk → {len(dfs)} selected '
      f'(MIN_TURNS={MIN_TURNS}, MAX_SESSIONS={MAX_SESSIONS}, NAME_FILTER={NAME_FILTER!r})')
for sid, name, n, df in candidates:
    print(f'  {name[:24]:24} {sid[:8]}  {n:4d} turns  max_ctx={df.ctx_pct.max():.1%}')

138 CSVs on disk → 12 selected (MIN_TURNS=30, MAX_SESSIONS=12, NAME_FILTER=None)
  find-doc                 c2c2efd3   648 turns  max_ctx=385.5%
  find-doc                 e68608c5   362 turns  max_ctx=191.3%
  find-doc                 645c3b26   310 turns  max_ctx=127.9%
  find-doc                 9f027ad3   230 turns  max_ctx=93.4%
  find-doc                 40de19e3   213 turns  max_ctx=86.2%
  find-doc                 c9e816d0   209 turns  max_ctx=80.3%
  find-doc                 05afca3f   194 turns  max_ctx=102.0%
  find-doc                 7a419704   190 turns  max_ctx=124.2%
  find-doc                 3d052dce   187 turns  max_ctx=78.7%
  finance                  fd59c881   180 turns  max_ctx=68.0%
  find-doc                 f3f57e1b   168 turns  max_ctx=86.8%
  claude-dynamic-island    3767a531   163 turns  max_ctx=55.4%


In [3]:
def composite_quality(df: pd.DataFrame) -> pd.Series:
    """
    Composite quality proxy (0–1, higher = better).
    Combines output density, penalises self-corrections and repetition.
    All components normalised to [0, 1] within the session.
    """
    def norm(s):
        r = s.max() - s.min()
        return (s - s.min()) / r if r > 0 else pd.Series(0.5, index=s.index)

    density_n   = norm(df['output_density'])
    sc_penalty  = norm(df['self_correction_count'])   # higher SC = lower quality
    rep_penalty = norm(df['repetition_score'])         # higher rep = lower quality

    return (density_n * 0.5 + (1 - sc_penalty) * 0.3 + (1 - rep_penalty) * 0.2).clip(0, 1)


def find_inflection(ctx: np.ndarray, quality: np.ndarray) -> float | None:
    """
    Estimates the inflection point as the ctx_pct where the quality
    slope (rolling linear regression) first turns consistently negative.
    Returns None if the session doesn't reach a clear inflection.
    """
    if len(ctx) < 6:
        return None
    window = max(3, len(ctx) // 5)
    slopes = []
    for i in range(window, len(ctx)):
        xi = ctx[i - window:i]
        yi = quality[i - window:i]
        slope = np.polyfit(xi, yi, 1)[0]
        slopes.append((ctx[i], slope))
    # First point where slope is negative for 2+ consecutive windows
    for i in range(1, len(slopes)):
        if slopes[i][1] < 0 and slopes[i - 1][1] < 0:
            return slopes[i - 1][0]
    return None

In [ ]:
def plot_session(ax, df: pd.DataFrame, session_id: str):
    df = df.sort_values('turn_index').reset_index(drop=True)
    df['quality'] = composite_quality(df)

    ctx = df['ctx_pct'].values
    q   = df['quality'].values

    # Smooth for display (preserve raw for scatter)
    q_smooth = savgol_filter(q, min(5, len(q) - (len(q) % 2 == 0)), 2) if len(q) >= 5 else q

    # Segment coloring
    for i in range(len(ctx) - 1):
        c = ctx[i]
        color = RED if c >= 0.80 else AMBER if c >= 0.60 else GREEN
        ax.plot(ctx[i:i+2], q_smooth[i:i+2], color=color, linewidth=2, solid_capstyle='round')
        ax.fill_between(ctx[i:i+2], q_smooth[i:i+2], alpha=0.06, color=color)

    # Raw scatter
    ax.scatter(ctx, q, s=20, zorder=5,
               c=[RED if c >= 0.80 else AMBER if c >= 0.60 else GREEN for c in ctx],
               alpha=0.7)

    # Threshold lines
    ax.axvline(0.60, color=AMBER, linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axvline(0.80, color=RED,   linewidth=0.8, linestyle='--', alpha=0.5)
    ax.text(0.61, 0.02, 'Soft GC', color=AMBER, fontsize=8, alpha=0.7)
    ax.text(0.81, 0.02, 'Hard GC', color=RED,   fontsize=8, alpha=0.7)

    # Inflection point
    inflection = find_inflection(ctx, q_smooth)
    if inflection:
        ax.axvline(inflection, color='#ffffff', linewidth=1.2, linestyle=':', alpha=0.6)
        ax.text(inflection + 0.01, 0.95, f'↓ {inflection:.0%}', color='#ffffff', fontsize=8, alpha=0.8)

    # Correlation
    if len(ctx) >= 4:
        r, p = pearsonr(ctx, q)
        ax.text(0.02, 0.95, f'r={r:.2f}', transform=ax.transAxes,
                fontsize=8, color=GREEN if r < -0.3 else DIM, va='top')

    ax.set_xlim(0, max(ctx.max() + 0.05, 0.15))
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel('context utilisation')
    ax.set_ylabel('quality proxy')
    ax.set_title(session_id[:24], color='#aeaeb2', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))


# `dfs` is the filtered selection built in the cell above.
if not dfs:
    raise SystemExit('No sessions matched the filters — lower MIN_TURNS or clear NAME_FILTER.')

n = len(dfs)
cols = min(3, n)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows), squeeze=False)

for idx, (sid, df) in enumerate(dfs.items()):
    ax = axes[idx // cols][idx % cols]
    plot_session(ax, df, sid)

# Hide unused subplots
for idx in range(n, rows * cols):
    axes[idx // cols][idx % cols].set_visible(False)

legend = [
    mpatches.Patch(color=GREEN, label='Clean (< 60%)'),
    mpatches.Patch(color=AMBER, label='Soft GC (60–80%)'),
    mpatches.Patch(color=RED,   label='Hard GC (> 80%)'),
]
fig.legend(handles=legend, loc='lower center', ncol=3, framealpha=0.2,
           facecolor='#1c1c1e', edgecolor='#3a3a3c', fontsize=9)

fig.suptitle('Context Efficiency Curves', fontsize=14, y=1.01, color='#f2f2f7')
plt.tight_layout()
plt.savefig('efficiency_curves.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0f')
plt.show()
print('Saved → analysis/efficiency_curves.png')

In [ ]:
# Summary stats across all sessions
rows = []
for sid, df in dfs.items():
    df = df.sort_values('turn_index').reset_index(drop=True)
    df['quality'] = composite_quality(df)
    ctx = df['ctx_pct'].values
    q   = df['quality'].values
    q_smooth = savgol_filter(q, min(5, len(q) - (len(q) % 2 == 0)), 2) if len(q) >= 5 else q
    r, _ = pearsonr(ctx, q) if len(ctx) >= 4 else (float('nan'), None)
    inflection = find_inflection(ctx, q_smooth)
    rows.append({
        'session':          sid[:16],
        'turns':            len(df),
        'max_ctx_pct':      f"{ctx.max():.1%}",
        'avg_output_tokens': df['output_tokens'].mean().round(0),
        'avg_sc_count':     df['self_correction_count'].mean().round(2),
        'avg_rep_score':    df['repetition_score'].mean().round(3),
        'quality_r':        round(r, 3),
        'inflection_ctx':   f"{inflection:.1%}" if inflection else '—',
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))